# Session 4 — Files, NumPy & Pandas I
### Block B · Saint Mary's General Hospital — Operations Analytics Unit

> *"Good morning. The CFO sent you the actual export from the hospital information system. Three files. Some of them messy. Tell us what we can see."*

**Yesterday** we built the hospital in Python — by hand. Today the data arrives from a real system, in CSV form, and we meet the two libraries every data scientist lives in: **NumPy** and **Pandas**.

By the end of this session you can:
1. **Load** CSV / Excel files robustly (encoding, separators, missing values).
2. Use **NumPy vectorisation** instead of Python loops.
3. Read and manipulate a **Pandas DataFrame** — filter, sort, fillna, basic descriptive stats.
4. Recognise the three classic data-loading bugs: encoding errors, silent NaNs, and `SettingWithCopyWarning`.

---

**Setting:** It's Thursday. The CFO drops three files on your desk at 08:55 and walks off. By 13:00 he wants to know: *which DRGs are losing us money, and what's our average occupancy?*

## §1 — Files in Python

A file on disk is just bytes. To read it as text, Python needs to know the **encoding**. To read it as a table, *something* needs to know the **separator**.

> Mental model: `open()` opens a door. `with` closes it for you when you leave the room.

In [ ]:
# The lo-fi way: just stdlib. We will only do this once, then move to pandas.
from pathlib import Path

# Robust: walk up the directory tree until we find a 'data/' folder.
# Works whether the notebook is opened from session4/, challenges/, or solutions/.
HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found above current notebook"
print("Looking in:", DATA)
print("Files:", sorted(p.name for p in DATA.glob("*")))

In [ ]:
# Reading a small CSV the manual way:
import csv

with open(DATA / "drg_catalog.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(len(rows), "DRG codes")
print(rows[0])

### Why `with`?

`with open(...) as f:` guarantees that the file is closed *even if your code crashes inside the block*. The alternative is `f = open(...); f.close()`, but if anything between those lines throws, you leak file handles.

A handful of patients per second is fine. A million open files is a server outage.

## §2 — Reading CSV with pandas

`pandas.read_csv()` is the Swiss Army knife. It:
- guesses the separator (often)
- parses numbers and dates (when asked)
- handles missing values (NaN)
- gives you a DataFrame — a labelled table

Almost every keyword argument you'll ever set to `read_csv` is *because some real-world file is broken in some specific way*.

In [ ]:
import pandas as pd

drg = pd.read_csv(DATA / "drg_catalog.csv")
drg.head()

In [ ]:
# .info() is the first thing you call on any new DataFrame:
drg.info()

In [ ]:
# .describe() gives you a numeric summary — useful for sanity checks:
drg.describe()

## §3 — NumPy: arrays as typed memory

A Python list can hold anything: numbers, strings, dicts, even other lists. That flexibility costs **memory** and **speed**.

A NumPy array is a slab of memory of one fixed type — usually a number type (int64, float64). That makes it small, fast, and well-suited for arithmetic.

> Mental model: a list is a notebook with words, drawings, and post-its. A NumPy array is graph paper with numbers in the squares — boring, but the calculator loves it.

In [ ]:
import numpy as np

# A list of lengths-of-stay (LOS) — Python list:
los_list = [5, 3, 9, 4, 7, 2, 8, 6, 1, 10]

# Same data as a NumPy array — typed, contiguous in memory:
los = np.array(los_list)
print(los, los.dtype)

In [ ]:
# Vectorisation: do something to every element WITHOUT a Python loop.
# This runs in C, fast.

print(los * 220)        # rough cost estimate per day
print(los > 5)          # boolean mask: which stays were long?
print(los.mean(), los.std())

In [ ]:
# Filtering with a mask — no for-loop, no if-statement:
long_stays = los[los > 5]
print(long_stays)        # only the elements where the mask was True

## §4 — Broadcasting: when shapes meet

NumPy operates element-wise. If two arrays have the same shape, that's straightforward. If shapes differ, NumPy *broadcasts* the smaller one across the larger.

This sounds abstract until you need it for real cost calculations.

In [ ]:
# Example: actual cost per case, vs. expected reimbursement.
actual_cost   = np.array([3450.00, 1820.00, 8950.00, 2380.00, 7800.00])
reimbursement = np.array([3450.00, 1820.00, 8950.00, 2380.00, 7800.00])

# Margin per case — vectorised, no loop:
margin = reimbursement - actual_cost
print("Margin per case:", margin)
print("Total margin:   ", margin.sum())
print("Loss-making cases:", (margin < 0).sum())

In [ ]:
# Broadcasting in action: apply a 5% efficiency target to ALL of them at once.
target_factor = 0.95
target_cost = reimbursement * target_factor   # one number broadcast across the array
print("Target cost per case:", target_cost)

## ⏸️ Mini-Challenge 1 — Load & Inspect (25 min)

Open `challenges/ch1_load_inspect.ipynb`.

**You will:** load `patients.csv`, inspect with `.head()`, `.info()`, `.describe()`, count patients with ≥ 2 chronic conditions. *Fast-Track:* compute the chronic-condition rate per insurance type.

## §5 — Pandas Series

A **Series** is a 1-dimensional NumPy array with **labels** (an index). Every column of a DataFrame is a Series.

> Mental model: a Series is a NumPy array that remembers what each cell is for.

In [ ]:
# Create a Series by hand:
ages = pd.Series([67, 54, 82, 39, 71], index=["P0001", "P0002", "P0003", "P0004", "P0005"])
print(ages)

In [ ]:
# Access by label, not position — meaning, like a dict, but vectorised under the hood:
print(ages["P0003"])
print(ages > 65)
print(ages[ages > 65])

# Vectorised arithmetic still works:
print((ages * 0.95).round(1))

## §6 — Pandas DataFrame

A **DataFrame** is a table — a dict of Series sharing the same row index.

> Mental model: it's the spreadsheet you would have built in Excel, but you can ask it questions in code.

In [ ]:
patients = pd.read_csv(DATA / "patients.csv")
patients.head()

In [ ]:
# Three things you do every time you open a new dataset:
print("Shape:", patients.shape)         # (rows, columns)
print("Columns:", list(patients.columns))
patients.dtypes

In [ ]:
# Filter — boolean indexing, just like NumPy:
elderly = patients[patients["age"] >= 65]
print(len(elderly), "elderly out of", len(patients))

In [ ]:
# Sort:
patients.sort_values("age", ascending=False).head(5)

In [ ]:
# Counting categories:
patients["primary_insurance"].value_counts()

### Handling missing values

Real CSVs have gaps. Pandas marks them with **NaN** (Not a Number). NaN never equals anything, even itself.

In [ ]:
# Detect missing values:
patients.isna().sum()      # how many NaN per column

# Two strategies:
# 1) drop rows with any NaN:        patients.dropna()
# 2) fill NaN with a value:         patients.fillna({"age": patients["age"].median()})


## ⏸️ Mini-Challenge 2 — Vectorised Margin (25 min)

Open `challenges/ch2_vectorised_margin.ipynb`.

**You will:** join encounters with the DRG catalog, compute margin = reimbursement − actual cost across 320 encounters using vectorised ops. *Fast-Track:* identify the DRGs that consistently lose money.

## §7 — Klinik: First look at 320 real encounters

The CFO wants two answers by 13:00:

1. **Average ward occupancy in Q1 2026.**
2. **Which DRGs lose us money and how often?**

We will load the encounters, join them with the DRG catalog, compute margins, and group by DRG. All vectorised, no Python loops.

In [ ]:
enc = pd.read_csv(DATA / "encounters.csv", parse_dates=["admission_date", "discharge_date"])
enc.head()

In [ ]:
print(f"{len(enc)} encounters across {enc['patient_id'].nunique()} unique patients")
print(f"Date range: {enc['admission_date'].min().date()} → {enc['discharge_date'].max().date()}")
print(f"Wards: {enc['ward'].unique()}")

In [ ]:
# Join with DRG catalog to get reimbursement per encounter
drg = pd.read_csv(DATA / "drg_catalog.csv")
enc = enc.merge(drg, on="drg_code", how="left")
enc[["encounter_id", "ward", "drg_code", "label", "actual_cost_eur", "base_reimbursement_eur"]].head()

In [ ]:
# Margin per case — one vectorised line:
enc["margin_eur"] = enc["base_reimbursement_eur"] - enc["actual_cost_eur"]
enc[["encounter_id", "drg_code", "actual_cost_eur", "base_reimbursement_eur", "margin_eur"]].head()

In [ ]:
# Loss-making cases:
losing = enc[enc["margin_eur"] < 0]
print(f"Loss-making cases: {len(losing)} of {len(enc)} ({len(losing)/len(enc):.1%})")
print(f"Total negative margin: € {losing['margin_eur'].sum():,.2f}")

In [ ]:
# Which DRGs are most often loss-makers? (preview of GroupBy — full course tomorrow)
loss_by_drg = (losing.groupby("drg_code")["margin_eur"]
                     .agg(["count", "sum"])
                     .sort_values("sum")
                     .head(5))
loss_by_drg

**One vectorised pipeline replaced 30 lines of yesterday's loops.** That's the entire pitch for Pandas.

## ⏸️ Mini-Challenge 3 — Clean a Messy Export (15 min)

Open `challenges/ch3_messy_csv.ipynb`. The file `messy_export.csv` is *intentionally* dirty — wrong encoding, semicolon separator, NaN, duplicates, weird types. Your job: load it, clean it, and report what's still broken.

## §8 — Differential diagnosis: three load-time bugs

| Symptom | Cause | Fix |
|---|---|---|
| `UnicodeDecodeError` while loading | wrong encoding | try `encoding="cp1252"` (Windows) or `"utf-8"` |
| Numbers come in as strings | comma decimal, or trailing space | `decimal=","`, then `.str.strip().astype(float)` |
| `SettingWithCopyWarning` | you modified a slice, not the original | use `df.loc[mask, "col"] = value` |
| All NaN in a column you thought was full | `,` was treated as decimal in the wrong locale | `pd.read_csv(..., decimal=",")` |
| `KeyError: 'wards'` (typo) | column name has trailing space, accent, or different case | `df.columns = df.columns.str.strip().str.lower()` |

---

## Take-home for tomorrow

> Open the CSV you brought from session 3 (or pick any tabular file you have). Load it with pandas. Print `.head()`, `.info()`, `.describe()`. Compute one *vectorised* expression that gives you a single business-relevant number. Bring the screenshot to session 5.

## Cliffhanger → Session 5 (Friday)

Tomorrow we go deeper into Pandas: **GroupBy** (what the CFO actually wanted), **Merge/Join** between tables, and **Visualisation** — turning numbers into pictures that a hospital board will read in 5 seconds.

We will also talk about **Clean Code**: how to structure your notebook so it's prüfungsreif on Tuesday.